In [ ]:
import os, glob

import cytoflow as flow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import ipywidgets as widgets
from IPython.display import display

from flow_analysis_tools import flow_violin
from flow_analysis_tools.color_palettes import *
from rclone_tools import NodeExperimentsDirectory

## Notebook and Experiment Setup

In [ ]:
DRIVE_RELPATH = "ARS_Working/Experiments/LX2-Cas12a-Col1a1-Enhancers/260910_c12c1a1R2_Promoter_Guide_Test/260914_c12a1a1R2_HEK_Fixed_Guide_GFP_Check"
DATA_DIR = "/data_gilbert3/asathler/hsiung-box"
PROXY = "http://169.230.134.132:3128"

In [ ]:
Experiment = NodeExperimentsDirectory(
    drive="hsiung", proxy=PROXY, output_dirname="out",
    source_relpath=DRIVE_RELPATH, node_mountpath=DATA_DIR,
)

In [ ]:
Experiment.stage()

In [ ]:
%matplotlib widget
matplotlib.rc('figure', dpi = 160)
plt.style.use('ggplot')

In [ ]:
FCS_FILES = sorted(glob.glob(os.path.join(Experiment.dest_dirpath, "*.fcs")))
YLIMITS = (0, 7)
CHANNELS = {
    "VL1-A": ["Empty", "None"],
    "BL1-A": ["Guide", "eGFP"],
    "RL1-A": ["Empty", "None"],
    "YL1-A": ["Empty", "None"]
}
CHANNEL_CMAPS = {
    "VL1-A": black_blue_pastel_palette,
    "BL1-A": black_green_pastel_palette,
    "RL1-A": black_red_pastel_palette,
    "YL1-A": black_yellow_pastel_palette,
}
ALL_NEG_CONSTRUCT = "HEK"
GATE_INFO = {}
print('\n'.join([f"'{x}'" for x in FCS_FILES]))
if isinstance(Experiment, NodeExperimentsDirectory):
    for x in FCS_FILES: Experiment.logger.info(x)

## Import Setup
Expects file names to have paired identity.

Each construct (e.g. LX2, Cas9, Cas12, pARS001, pARS002) should have 1 or 2 corresponding "guides" (or downstream conditions) assigned to them. So:
- **_LX2_**: Stained & Unstained
- **_Cas9_**: sgNT and sgTargeting
- ...

In [ ]:
NAME_N_UNDERSCORE = 1
def get_sample_name(name):
    if "." in name: name = os.path.splitext(name)[0]
    return name.split("_")[-NAME_N_UNDERSCORE]

In [ ]:
FCS_NAMES = [os.path.basename(filename) for filename in FCS_FILES]
SAMPLE_NAMES = [get_sample_name(name) for name in FCS_NAMES]
max_fcs_len = max([len(x) for x in FCS_NAMES]) + 5
for fcs, sample_name in zip(FCS_NAMES, SAMPLE_NAMES):
    print_txt = f"{fcs + ''.join([' '] * (max_fcs_len - len(fcs)))}{str(sample_name):<30}"
    print(print_txt)
    if isinstance(Experiment, NodeExperimentsDirectory):
        Experiment.logger.info(print_txt)

import_op = flow.ImportOp(
    conditions = {
        "Construct": "category",
    },
    tubes = [
        flow.Tube(file=fcs_file, conditions={
            "Construct": get_sample_name(fcs_file),
    }) for fcs_file in FCS_FILES],
    # channels = channel_names
)

try:
    ex = import_op.apply()
except ValueError as e:
    error_text = str(e)
    if error_text == "Categorical categories cannot be null":
        raise Exception(
            "Functions 'get_construct_name' or 'get_guide_type' likely are turning None or NoneType." + \
            "\n\t   This is typically because a construct or guide name case isn't properly handled."
        )
    else:
        raise e

## Select Cells

In [ ]:
single_gate = flow.PolygonOp(
    name="Single_Cell",
    xchannel="FSC-A",
    ychannel="SSC-A"
)

single_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True
).plot(ex, gridsize = 100)

In [ ]:
if isinstance(Experiment, NodeExperimentsDirectory):
    plt.savefig(Experiment.output_dirpath / f"single_cell_gate.png")

In [ ]:
GATE_INFO["single_cell_vertices"] = np.asarray(single_gate.vertices)
ex = single_gate.apply(ex)
plt.close()

## Gating

### Gating Template
Use the template below for help with gating

##### Quad Gate
    quad_gate = flow.QuadOp(
        name="",
        xchannel="",
        ychannel="",
    )

##### Single gate
    single_gate = flow.PolygonOp(
        name="",
        xchannel="",
        ychannel="",
        yscale="logicle"
    )

##### Drawing / Applying Gate
    ___gate = flow.PolygonOp(name="") # placeholder variable
    ___gate.default_view(
        density = True,
        huescale = "log",
        interactive = True,
        subset = "Single_Cell == True"
    ) #.plot(ex_single, gridsize=100)
    ex_single_gated = ___gate.apply(ex_single)
    plt.close()


In [ ]:
plt.ioff()

channels = list(CHANNELS)
subset = f"(Single_Cell == True) and (Construct == '{ALL_NEG_CONSTRUCT}')"
gates = {}

out = widgets.Output()
status = widgets.Label()
btn = widgets.Button(description="Accept threshold", button_style="success")

# Keep strong references: matplotlib holds event callbacks weakly, so if the
# view is garbage-collected, its click handler silently disappears.
state = {"i": 0, "op": None, "view": None, "fig": None}

def show_channel(i):
    chan = channels[i]
    op = flow.ThresholdOp(name=chan, channel=chan)
    view = op.default_view(huescale="log", interactive=True,
                           subset=subset, scale="log")
    plt.close("all")
    with out:
        out.clear_output(wait=True)
        view.plot(ex)
        fig = plt.gcf()
        display(fig.canvas)

    state.update(op=op, view=view, fig=fig)
    status.value = f"[{i+1}/{len(channels)}] {chan}: click the plot, then Accept"

def on_accept(_b):
    op = state["op"]
    if op.threshold is None:
        status.value = "No threshold set yet; click the plot first"
        return
    gates[op.channel] = op
    print(f"{op.channel}: {op.threshold}")
    state["i"] += 1
    if state["i"] < len(channels):
        show_channel(state["i"])
    else:
        plt.close(state["fig"])
        btn.disabled = True
        status.value = f"Done: {len(gates)} gates stored in `gates`"

btn.on_click(on_accept)
display(widgets.VBox([out, widgets.HBox([btn, status])]))
show_channel(0)
print(len(state["fig"].axes))

In [ ]:
for chan, thresh_op in gates.items():
    GATE_INFO[f"{chan}_threshold"] = thresh_op.threshold
    for char in ["-", "_"]:
        thresh_op.name = thresh_op.name.replace(char, "")
    ex = thresh_op.apply(ex)

In [ ]:
if isinstance(Experiment, NodeExperimentsDirectory):
    ex.data.to_csv(os.path.join(Experiment.output_dirpath, "processed_data.csv"))

plot_data = ex.data[ex.data["Single_Cell"]==True]

# The following lines are to prevent NANs being fed into log y axes
# They may not be necessary with newer versions of Cytoflow (>1.2.2)
num_cols = plot_data.select_dtypes(include='number').columns
plot_data[num_cols] = plot_data[num_cols].replace(0, 1)

## Graphing

In [ ]:
%matplotlib inline

In [ ]:
for flow_channel, fluor in CHANNELS.items():
    fluor_label = f"{fluor[0]} ({fluor[1]})"

    # See: https://matplotlib.org/stable/gallery/lines_bars_and_markers/scatter_hist.html#id1
    fig, axs = plt.subplot_mosaic(
        [
            ["Mock", "Construct"]
        ],
        figsize=(8,4),
        width_ratios=(1,4),
        sharey=True
        # layout="
        # consrained"
    )

    flow_violin(
        data=plot_data.loc[(plot_data["Construct"] == ALL_NEG_CONSTRUCT)],
        x="Construct",
        y=flow_channel,
        color=CHANNEL_CMAPS[flow_channel][1],
        log_y_axis=True,
        order=[ALL_NEG_CONSTRUCT],
        ylabel=fluor_label,
        xlabel="",
        density_norm='width',
        ylim=YLIMITS,
        ax=axs['Mock']
    )

    flow_violin(
        data=plot_data.loc[(plot_data["Construct"] != ALL_NEG_CONSTRUCT) & (plot_data["Construct"] != "Unstained")],
        x="Construct",
        y=flow_channel,
        color=CHANNEL_CMAPS[flow_channel][1],
        log_y_axis=True,
        xlabel="Epigenetic Construct (Constitutive)",
        ylabel="",
        ylim=YLIMITS,
        density_norm='width',
        ax=axs['Construct']
    )

    axs['Mock'].legend(title="Control", loc="upper right")
    axs['Construct'].legend(title="Construct", loc="upper right")
    axs['Construct'].set_xticklabels(axs['Construct'].get_xticklabels(), rotation=12.25)
    fig.subplots_adjust()
    if isinstance(Experiment, NodeExperimentsDirectory):
        imgpath = os.path.join(Experiment.output_dirpath, f"{flow_channel}.png")
        msg = f"Image of {flow_channel} [{fluor}] saved at {imgpath}"
        plt.savefig(imgpath)
        Experiment.logger.info(msg)

In [ ]:
if isinstance(Experiment, )
    savez_path = Experiment.output_dirpath / f"{Experiment.experiment_name}_gating.npz"
    msg = f"Saving gating data to {savez_path}"
    np.savez(file=savez_path, **GATE_INFO)
    Experiment.logger.info(msg)

In [ ]:
Experiment.close()